# Phase6: Factor Combination

In [ ]:
def daily_rank_ic(alpha_df, fwd_ret_df):
    alpha, ret = alpha_df.align(fwd_ret_df, join="inner", axis=0)
    alpha, ret = alpha.align(ret, join="inner", axis=1)

    a_rank = alpha.rank(axis=1)
    r_rank = ret.rank(axis=1)

    ic_series = a_rank.corrwith(r_rank, axis=1)
    return ic_series

In [1]:
def ic_summary(ic_series):
    ic_mean = ic_series.mean()
    ic_std  = ic_series.std(ddof=1)
    ic_ir   = ic_mean / ic_std if ic_std > 0 else np.nan
    t_stat  = ic_mean / (ic_std / sqrt(ic_series.count())) if ic_std > 0 else np.nan
    return {
        "IC_mean": ic_mean,
        "IC_std": ic_std,
        "IC_IR": ic_ir,
        "t_stat": t_stat
    }

In [2]:
alphas = {
    "0617a": alpha_pv1,
    "0810a": alpha2,
    "0415b": alpha3,
    "0812a": alpha5,
    "0612d": alpha6,
    "0608d": alpha7,
    "0615a": alpha8,
    "0604c": alpha9,
    "0407a": alpha10,
    "0413a": alpha12,
    "0413b": alpha13
}

ic_stats = {}
ic_series_dict = {}

for name, a in alphas.items():
    ic = daily_rank_ic(a, fwd_ret)
    ic_series_dict[name] = ic
    ic_stats[name] = ic_summary(ic)

pd.DataFrame(ic_stats).T.sort_values("IC_mean", ascending=False)

NameError: name 'alpha_pv1' is not defined

In [3]:
def flatten_panel(df):
    return df.stack(dropna=True)

alpha_flat = {k: flatten_panel(v) for k, v in alphas.items()}
alpha_flat_df = pd.DataFrame(alpha_flat)

signal_corr = alpha_flat_df.corr()
signal_corr

NameError: name 'alphas' is not defined

In [ ]:
ic_df = pd.DataFrame(ic_series_dict) 
ic_corr = ic_df.corr()
ic_corr

---

In [ ]:
import numpy as np
import pandas as pd

# ---------- helpers ----------
def winsorize_cs(df, lower=0.01, upper=0.01):
    lo = df.quantile(lower, axis=1)
    hi = df.quantile(1 - upper, axis=1)
    v = df.to_numpy(dtype=float)
    v = np.where(~np.isnan(lo.to_numpy()[:, None]), np.maximum(v, lo.to_numpy()[:, None]), v)
    v = np.where(~np.isnan(hi.to_numpy()[:, None]), np.minimum(v, hi.to_numpy()[:, None]), v)
    return pd.DataFrame(v, index=df.index, columns=df.columns)

def zscore_cs(df):
    mu = df.mean(axis=1)
    sd = df.std(axis=1, ddof=0).replace(0, np.nan)
    return df.sub(mu, axis=0).div(sd, axis=0)

def norm_l1(df):
    s = df.abs().sum(axis=1).replace(0, np.nan)
    return df.div(s, axis=0).fillna(0.0)

def signal_to_weight(sig):
    return norm_l1(zscore_cs(winsorize_cs(sig)))

def alpha_ret_series(w, fwd):
    fr = fwd.reindex(index=w.index, columns=w.columns)
    return (w * fr).sum(axis=1)

def daily_rank_ic(alpha_df, fwd_df):
    a, r = alpha_df.align(fwd_df, join="inner", axis=0)
    a, r = a.align(r, join="inner", axis=1)
    return a.rank(axis=1).corrwith(r.rank(axis=1), axis=1)

def combine_alphas_to_weights(a_t, all_weights):
    names = [c for c in a_t.columns if c in all_weights]
    inst = sorted(set().union(*[set(all_weights[k].columns) for k in names]))
    out = pd.DataFrame(0.0, index=a_t.index, columns=inst)
    for k in names:
        wk = all_weights[k].reindex(index=a_t.index, columns=inst).fillna(0.0)
        out = out.add(wk.mul(a_t[k], axis=0), fill_value=0.0)
    return out

# ---------- build common objects ----------
alpha_names = list(alphas.keys())
all_weights = {k: signal_to_weight(alphas[k]) for k in alpha_names}
alpha_returns_df = pd.DataFrame({k: alpha_ret_series(all_weights[k], fwd_ret) for k in alpha_names}).sort_index()

In [ ]:
# Equal
K = len(alpha_names)
a_equal = pd.DataFrame(1.0 / K, index=alpha_returns_df.index, columns=alpha_names)
a_equal = norm_l1(a_equal)

# IC (rolling mean IC, lagged)
ic_df = pd.DataFrame({k: daily_rank_ic(alphas[k], fwd_ret) for k in alpha_names}).reindex(alpha_returns_df.index)
ic_score = ic_df.shift(1).rolling(63, min_periods=20).mean()
a_ic = norm_l1(ic_score)

# MVO (rolling mu/cov on lagged alpha returns)
R_info = alpha_returns_df.shift(1)  # no lookahead
lam, ridge, gamma = 15.0, 1e-3, 1.0
I = np.eye(K)
a_prev = np.zeros(K)
rows = []

for t, dt in enumerate(R_info.index):
    hist = R_info.iloc[max(0, t-62):t+1].dropna(how="all")
    if len(hist) < 20:
        mu = np.zeros(K)
        S = I.copy()
    else:
        mu = hist.mean().reindex(alpha_names).fillna(0.0).to_numpy()
        cov = hist.cov().reindex(index=alpha_names, columns=alpha_names).fillna(0.0).to_numpy()
        diag = np.diag(np.diag(cov))
        S = 0.8 * cov + 0.2 * diag
        S = 0.5 * (S + S.T)

    Q = 2.0 * (lam * S + ridge * I + gamma * I)
    b = mu + 2.0 * gamma * a_prev
    try:
        a = np.linalg.solve(Q, b)
    except np.linalg.LinAlgError:
        a = np.linalg.pinv(Q) @ b

    a_prev = np.nan_to_num(a, nan=0.0)
    rows.append(pd.Series(a_prev, index=alpha_names, name=dt))

a_mvo = norm_l1(pd.DataFrame(rows))


In [ ]:
w_equal = combine_alphas_to_weights(a_equal, all_weights)
w_ic    = combine_alphas_to_weights(a_ic, all_weights)
w_mvo   = combine_alphas_to_weights(a_mvo, all_weights)

r_equal = alpha_ret_series(w_equal, fwd_ret)
r_ic    = alpha_ret_series(w_ic, fwd_ret)
r_mvo   = alpha_ret_series(w_mvo, fwd_ret)

eq = pd.DataFrame({
    "equal": (1 + r_equal.fillna(0)).cumprod(),
    "ic":    (1 + r_ic.fillna(0)).cumprod(),
    "mvo":   (1 + r_mvo.fillna(0)).cumprod(),
})
eq.plot(figsize=(10,4), title="Equal vs IC vs MVO")
